# Active learning for 2D classification

As a bonus, we show active learning using the posterior uncertainty on a
two-dimensional classification with three classes.

First, we define the ground truth decision boundary function.

In [ ]:
from functools import partial

import jax
import optax
from flax import nnx
from helper import DataLoader, Model, split, train_model
from jax import numpy as jnp
from jax import random
from matplotlib import pyplot as plt
from optax.losses import softmax_cross_entropy_with_integer_labels
from plotting import (
    plot_datapoints,
    plot_decision_boundaries,
    plot_next_point,
    plot_prediction,
    show_animation_classification,
)
from tqdm import tqdm

from laplax.curv import create_ggn_mv, create_posterior_fn
from laplax.eval.pushforward import (
    lin_pred_mean,
    lin_pred_std,
    lin_setup,
    set_lin_pushforward,
)

seed = 2392385
key = random.key(seed)
active_data_key, passive_data_key, sampling_key = random.split(key, 3)

In [ ]:
@jax.jit
def true_function(point):
    def f1(x):
        return 1.9 * x**3 - 1.5 * x**2 + 0.5

    def f2(x):
        return -1.5 * x**2 + 2 * x + 0.2

    x, y = point[0], point[1]
    return jnp.where(y >= f2(x), 2, jnp.where(y >= f1(x), 1, 0))

We generate some initial datapoints and visualize them.

In [ ]:
n_initial_datapoints = 20
key1, key2 = random.split(active_data_key)
xs = random.uniform(key1, shape=n_initial_datapoints, minval=0, maxval=1)
ys = random.uniform(key2, shape=n_initial_datapoints, minval=0, maxval=1)
points = jnp.stack((xs, ys)).mT
labels = jax.vmap(true_function)(points)
class_dataloader = DataLoader(points, labels, batch_size=10)

plot_decision_boundaries()
plot_datapoints(xs, ys, labels)
plt.show()

We reuse the model from above, a small fully connected network with three layers.
The only differences are the 2d input, 3 output logits
and cross entropy loss instead of MSE.

In [ ]:
@nnx.jit
def train_step(model, optimizer, batch, labels):
    def loss_fn(model):
        logits = model(batch)
        return softmax_cross_entropy_with_integer_labels(logits, labels).sum()

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss


start_model = Model(
    in_channels=2, hidden_channels=32, out_channels=3, rngs=nnx.Rngs(seed)
)

params = nnx.state(start_model)
total_params = sum(p.size for p in jax.tree.leaves(params))
print(f"Total number of parameters: {total_params}")

lr = 1e-3
class_optimizer = nnx.Optimizer(start_model, optax.adam(lr))
class_model = train_model(
    start_model, class_optimizer, class_dataloader, train_step, n_epochs=1000
)

We visualize the trained model's predictions as background color in the data plane.

In [ ]:
xv, yv = jnp.meshgrid(jnp.linspace(0, 1, 100), jnp.linspace(0, 1, 100))
points = jnp.stack([xv.ravel(), yv.ravel()], axis=-1)
logits = jax.vmap(class_model)(points)
preds = logits.argmax(axis=-1)

plot_decision_boundaries()
plot_datapoints(xs, ys, labels)
plot_prediction(preds)

In this example, we use only the first rule for maximal total information gain.
The maximum of the total information gain is at the same location as the maximum of
the standard deviation, as the constants and logarithm in the formula do not change
the location of the maximum.
Therefore, it is sufficient here to calculate the posterior uncertainty using laplax.
This is under the assumption that the prior precision is constant.

In [ ]:
def compute_uncertainty(model, dataloader):
    trainset = {"input": dataloader.X, "target": dataloader.y}
    model_fn, params = split(model)
    ggn_mv = create_ggn_mv(
        model_fn,
        params,
        trainset,
        loss_fn="cross_entropy",
    )
    posterior_fn = create_posterior_fn(
        curv_type="full",
        mv=ggn_mv,
        layout=params,
    )
    prob_predictive = partial(
        set_lin_pushforward,
        model_fn=model_fn,
        mean_params=params,
        posterior_fn=posterior_fn,
        pushforward_fns=[
            lin_setup,
            lin_pred_mean,
            lin_pred_std,
        ],
    )
    prior_arguments = {"prior_prec": 100.0}
    prob_predictive = prob_predictive(
        prior_arguments=prior_arguments,
    )
    pred = jax.vmap(prob_predictive)(points)
    return pred["pred_std"]


uncertainties = compute_uncertainty(class_model, class_dataloader)
uncertainty = uncertainties[jnp.arange(10000), preds]

We calculate the uncertainty on a regular grid within data space,
and find its maximum on the grid.
This is going to be the best next datapoint location, according to MacKay.
We visualize the uncertainty as the alpha-value of the prediction colors,
with stronger color corresponding to larger uncertainty.

'get_next_point_sampled' is an alternative rule to find the next datapoint,
which samples from the data plane by interpreting the uncertainty as logits to
a categorical distribution. This way, random points with high uncertainty are chosen,
which prevents the active learning loop from sampling in the same region repeatedly.
Feel free to try out both methods in the active learning loop and see the difference!


In [ ]:
def get_next_point(uncertainty):
    return points[jnp.argmax(uncertainty)]


def get_next_point_sampled(key, uncertainty):
    return points[jax.random.categorical(key, uncertainty)]


next_point = get_next_point(uncertainty)

plot_decision_boundaries()
plot_datapoints(xs, ys, labels)
plot_prediction(preds, uncertainty)
plot_next_point(next_point)

We see that the uncertainty is large where the model thinks the
decision boundary lies, and low elsewhere.
This means the active learning loop is going to sample in these areas,
confirming or adapting the found decision boundary.

In [ ]:
learning_rounds = 20
epochs_per_learning_round = 50
plot_data = []
sampling_keys = jax.random.split(sampling_key, learning_rounds)

for i, key in tqdm(enumerate(sampling_keys)):
    print(f"Active learning round {i + 1}")
    # 1) Sample new datapoint
    next_target = true_function(next_point)
    class_dataloader = class_dataloader.add(next_point, jnp.atleast_1d(next_target))

    # 2) Continue training
    class_model = train_model(
        class_model,
        class_optimizer,
        class_dataloader,
        train_step,
        n_epochs=epochs_per_learning_round,
    )
    grid_preds = jnp.argmax(class_model(points), axis=-1)

    # 3) Compute uncertainty
    uncertainties = compute_uncertainty(class_model, class_dataloader)
    uncertainty = uncertainties[jnp.arange(10000), grid_preds]

    # 4) Find next datapoint location
    next_point = get_next_point_sampled(key, uncertainty)
    # next_point = get_next_point(uncertainty)

    # Plotting
    data_preds = jnp.argmax(class_model(class_dataloader.X), axis=-1)
    plot_data.append((
        grid_preds,
        class_dataloader,
        data_preds,
        uncertainty,
        next_point,
    ))
    print("-----------------------")

In [ ]:
show_animation_classification(plot_data)

### Comparison against passive learning

As in the previous task, we compare our actively trained model against one
that is trained as usual, with a fixed dataset.

In [ ]:
n_passive_datapoints = n_initial_datapoints + learning_rounds
key1, key2 = random.split(passive_data_key)
xs = random.uniform(key1, shape=n_passive_datapoints, minval=0, maxval=1)
ys = random.uniform(key2, shape=n_passive_datapoints, minval=0, maxval=1)
datapoints = jnp.stack((xs, ys)).mT
labels = jax.vmap(true_function)(datapoints)
passive_class_dl = DataLoader(datapoints, labels, batch_size=10)

passive_class_model = Model(
    in_channels=2, hidden_channels=32, out_channels=3, rngs=nnx.Rngs(seed)
)
passive_class_optimizer = nnx.Optimizer(passive_class_model, optax.adam(lr))

passive_class_model = train_model(
    passive_class_model,
    passive_class_optimizer,
    passive_class_dl,
    train_step,
    n_epochs=2000,
)

logits = jax.vmap(passive_class_model)(points)
preds = logits.argmax(axis=-1)

plot_decision_boundaries()
plot_datapoints(xs, ys, labels)
plot_prediction(preds)
plt.show()

The passively trained model's prediction boundaries are not well-aligned with
the ground truth, simply because there are few datapoints close to the
ground truth boundaries, from which it could have learned them.